# 🎯 Pointwise Classical Models Benchmark: Complete Feature Progression

Evaluates all 8 unsupervised pointwise classical models across the complete feature progression:
1. **Raw Coordinates (6)**: `latitude`, `longitude`, `course`, `ground_speed`, `vertical_speed`, `height`
2. **Raw No Coordinates (4)**: `course`, `ground_speed`, `vertical_speed`, `height`
3. **Raw Engineered (8)**: `course`, `ground_speed`, `vertical_speed`, `height`, `acceleration`, `vertical_acceleration`, `turn_rate`, `path_curvature`
4. **Pure Kinematics Baseline (8)**: `height`, `ground_speed`, `vertical_speed`, `acceleration`, `turn_rate`, `path_curvature`, `heading_speed_consistency`, `motion_smoothness`
5. **Standard Baseline (10)**: Kinematics 8 + `prediction_error`, `yaw_acceleration`
6. **Noise Texture (13)**: Baseline 10 + `prediction_error_autocorrelation`, `position_residual_std`, `speed_spectral_entropy`
7. **Baseline + Cross-Correlation (13)**: Baseline 10 + `corr_speed_turn`, `corr_accel_turn`, `corr_vert_speed`
8. **Full Cross-Correlation & Noise Texture (16)**: Noise Texture 13 + 3 Cross-Correlations

### Models Evaluated (8 Total):
- `Isolation Forest`, `GMM`, `Mahalanobis`, `One-Class SVM`, `PCA`, `K-Means`, `DBSCAN`, `KNN`

In [ ]:
from pathlib import Path
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

PROJECT_ROOT = Path("..").resolve()
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from presets.run_unsupervised_pipeline import FEATURE_SETS, run_experiment
from implement.utils.helper import get_output_dir

print(f"Operating Directory: {os.getcwd()}")
print(f"Available Feature Presets: {list(FEATURE_SETS.keys())}")

## 1. Raw Coordinates (6 Features)

In [ ]:
df_raw_coords = run_experiment(
    exp_name="pointwise_raw_coords",
    feature_list=FEATURE_SETS["raw_coords"],
    model_family="pointwise",
    fresh_cache=True
)

## 2. Raw No Coordinates (4 Features)

In [ ]:
df_raw_no_coords = run_experiment(
    exp_name="pointwise_raw_no_coords",
    feature_list=FEATURE_SETS["raw_no_coords"],
    model_family="pointwise",
    fresh_cache=True
)

## 3. Raw Engineered (8 Features)

In [ ]:
df_raw_eng = run_experiment(
    exp_name="pointwise_raw_engineered",
    feature_list=FEATURE_SETS["raw_engineered"],
    model_family="pointwise",
    fresh_cache=True
)

## 4. Pure Kinematics Baseline (8 Features — without Yaw Accel & PE)

In [ ]:
df_b8 = run_experiment(
    exp_name="pointwise_baseline_8",
    feature_list=FEATURE_SETS["baseline_8"],
    model_family="pointwise",
    fresh_cache=True
)

## 5. Standard Baseline (10 Features)

In [ ]:
df_b10 = run_experiment(
    exp_name="pointwise_baseline_10",
    feature_list=FEATURE_SETS["baseline_10"],
    model_family="pointwise",
    fresh_cache=True
)

## 6. Noise Texture (13 Features)

In [ ]:
df_nt13 = run_experiment(
    exp_name="pointwise_noise_texture_13",
    feature_list=FEATURE_SETS["noise_texture_13"],
    model_family="pointwise",
    fresh_cache=True
)

## 7. Baseline + Cross-Correlation (13 Features)

In [ ]:
df_bc13 = run_experiment(
    exp_name="pointwise_baseline_corr_13",
    feature_list=FEATURE_SETS["baseline_corr_13"],
    model_family="pointwise",
    fresh_cache=True
)

## 8. Full Cross-Correlation & Noise Texture (16 Features)

In [ ]:
df_cc16 = run_experiment(
    exp_name="pointwise_correlation_16",
    feature_list=FEATURE_SETS["correlation_16"],
    model_family="pointwise",
    fresh_cache=True
)

## 9. Comprehensive Pointwise Aggregate Summary

In [ ]:
pw_experiments = [
    ("Raw Coords (6)", "pointwise_raw_coords"),
    ("Raw No Coords (4)", "pointwise_raw_no_coords"),
    ("Raw Engineered (8)", "pointwise_raw_engineered"),
    ("Baseline Kinematics (8)", "pointwise_baseline_8"),
    ("Baseline Standard (10)", "pointwise_baseline_10"),
    ("Noise Texture (13)", "pointwise_noise_texture_13"),
    ("Baseline + Corr (13)", "pointwise_baseline_corr_13"),
    ("Cross-Correlation (16)", "pointwise_correlation_16"),
]

agg_pw = []
for label, exp in pw_experiments:
    p = get_output_dir() / "pipeline_experiments" / exp / f"{exp}_aggregate.csv"
    if p.exists():
        df = pd.read_csv(p)
        df.insert(0, "Feature Preset", label)
        agg_pw.append(df)

if agg_pw:
    summary_df = pd.concat(agg_pw, ignore_index=True)
    display(summary_df)
    save_path = get_output_dir() / "pointwise_all_feature_progressions_aggregate.csv"
    summary_df.to_csv(save_path, index=False)
    print(f"Saved summary to {save_path}")